# IberoamericaBooks — reproducible portfolio demo

This notebook makes the complete ETL workflow visible step by step. It executes the same reusable pipeline used by the Streamlit app, so the technical walkthrough and the interactive demo cannot diverge.

Bibliographic fields are public demo metadata; sales and digital-attention variables are deterministic and synthetic. No private operational records are included.

## ETL workflow

1. **Ingestion** — read three Excel catalogues with different schemas.
2. **Normalization** — standardize ISBNs, titles and provenance while retaining raw values.
3. **Validation** — reject records whose ISBN-13 checksum is invalid.
4. **Deduplication** — merge editions found in more than one source using a deterministic priority.
5. **Enrichment** — attach synthetic sales and digital-attention metrics by ISBN.
6. **Persistence** — write the normalized relational model, foreign keys, traceability records and summary view to SQLite.

In [ ]:
from pathlib import Path
import sys, tempfile
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from iberoamerica_books.pipeline import run_pipeline
from iberoamerica_books.database import connect


## 1. Execute the reproducible pipeline

The orchestration stays in `src/iberoamerica_books/pipeline.py`; the notebook exposes its inputs and intermediate results instead of duplicating the production logic.

### Run the orchestration and inspect summary metrics

In [ ]:
database_path = Path(tempfile.mkdtemp()) / 'iberoamerica_books_demo.sqlite'
result = run_pipeline(ROOT / 'demo_data', database_path)
result.metrics

## 2. Inspect traceability and normalization

Every input row keeps its source, original position, raw values, normalized values and acceptance status.

In [ ]:
result.source_records[[
    'source_name', 'source_row', 'raw_isbn', 'isbn13',
    'raw_title', 'normalized_title', 'valid_isbn'
]].head(12)

## 3. Inspect the deduplicated catalogue

In [ ]:
result.catalogue[[
    'isbn13', 'title', 'authors', 'publisher', 'year',
    'country', 'source_count', 'source_names'
]]

## 4. Inspect quality controls and query SQLite

The pipeline event log exposes the input and output row count of every ETL stage before the final relational view is queried.

In [ ]:
result.events

In [ ]:
import pandas as pd
with connect(database_path) as connection:
    display(pd.read_sql_query('SELECT * FROM book_summary ORDER BY revenue_eur DESC', connection))

## What this demonstrates

- `result.source_records` preserves raw and normalized values for traceability.
- `result.events` documents row counts at every stage.
- `database_path` is a complete SQLite database that can be opened with any SQLite client.